## Step 1: Install and Import Libraries

We only need OpenAI for embeddings and NumPy for calculations.

In [17]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 2: Create Sample Q&A Cache

Let's create a simple cache with 5 question-answer pairs.

In [18]:
# HelpParcel Cached FAQ Dataset

cache = [
    {
        "question": "Where is my order?",
        "answer": "You can track your order by entering your tracking number on the Track Order page. Tracking updates may take up to 24-48 hours after shipment."
    },
    {
        "question": "How long does delivery take?",
        "answer": "Standard shipping typically takes 3-7 business days, while express shipping takes 1-3 business days depending on your location."
    },
    {
        "question": "Can I cancel my order?",
        "answer": "Orders can only be canceled before they are shipped. Once shipped, cancellation is no longer possible, but you may request a return after delivery."
    },
    {
        "question": "How do I request a refund?",
        "answer": "To request a refund, visit your order details page and select 'Request Refund'. Refunds are processed after the returned item passes inspection."
    },
    {
        "question": "My item arrived damaged. What should I do?",
        "answer": "Please submit photos of the damaged item within 48 hours of delivery. Our support team will review the case and arrange a replacement or refund."
    },
    {
        "question": "Can I change my shipping address?",
        "answer": "Shipping addresses can only be modified before the order is shipped. Visit your order page and select 'Edit Address' if available."
    },
    {
        "question": "Why is my tracking number not updating?",
        "answer": "Tracking updates may take 24-48 hours after shipment. If there are no updates after 5 business days, please contact support."
    },
    {
        "question": "Do you offer replacements?",
        "answer": "Yes, replacements are available for damaged, defective, or incorrect items reported within 7 days of delivery."
    },
    {
        "question": "Where can I find my order ID?",
        "answer": "Your order ID is available in your order confirmation email and in the 'My Orders' section of your account."
    },
    {
        "question": "What courier do you use?",
        "answer": "We work with multiple shipping partners depending on the destination. The assigned courier will appear in your order tracking details."
    }
]

## Step 3: Create Embeddings for Cached Questions

Convert each question into a vector (embedding) using OpenAI.

In [20]:
models = ["text-embedding-ada-002", "text-embedding-3-small", "text-embedding-3-large"]

def get_embedding(text):
    """Get embedding for a text using OpenAI."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding)

# Create embeddings for all cached questions
print("🔢 Creating embeddings for cached questions...")
for pair in cache:
    pair['embedding'] = get_embedding(pair['question'])
    
# print(f"✅ Created {len(cache)} embeddings")
# print(f"   Embedding dimension: {len(cache[0]['embedding'])}")
# print(f"   First few values: {cache[0]['embedding'][:5]}")
print(f"   Updated cache list: {cache}")

🔢 Creating embeddings for cached questions...


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## Step 4: Define Similarity Functions

We'll use cosine distance to measure how similar two questions are.

In [ ]:
def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

def cosine_distance(a, b):
    """Calculate cosine distance (1 - similarity)."""
    return 1 - cosine_similarity(a, b)

# Test with two identical questions
test_embedding = cache[0]['embedding']
distance = cosine_distance(test_embedding, test_embedding)
print(f"✅ Functions defined!")
print(f"   Distance between identical questions: {distance:.6f}")
print(f"   (Should be ~0.0)")

✅ Functions defined!
   Distance between identical questions: -0.000000
   (Should be ~0.0)


## Step 5: Implement Cache Lookup Function

This function searches the cache for similar questions.

In [ ]:
def search_cache(query, threshold=0.3):
    """
    Search cache for similar questions.
    
    Args:
        query: User's question
        threshold: Maximum distance for a match (lower = stricter)
    
    Returns:
        dict with match info or None
    """
    # Get embedding for the query
    query_embedding = get_embedding(query)
    
    # Calculate distances to all cached questions
    results = []
    for pair in cache:
        distance = cosine_distance(query_embedding, pair['embedding'])
        similarity = 1 - distance
        results.append({
            'question': pair['question'],
            'answer': pair['answer'],
            'distance': distance,
            'similarity': similarity
        })
    
    # Sort by distance (closest first)
    results.sort(key=lambda x: x['distance'])
    
    # Return best match if within threshold
    best_match = results[0]
    if best_match['distance'] <= threshold:
        return {
            'hit': True,
            'matched_question': best_match['question'],
            'answer': best_match['answer'],
            'distance': best_match['distance'],
            'similarity': best_match['similarity'],
            'all_distances': [(r['question'], r['distance']) for r in results]
        }
    else:
        return {
            'hit': False,
            'best_match': best_match['question'],
            'distance': best_match['distance'],
            'similarity': best_match['similarity'],
            'all_distances': [(r['question'], r['distance']) for r in results]
        }

print("✅ Cache search function ready!")

✅ Cache search function ready!


## Step 6: Test Cache Hits (Exact and Semantic Matches)

Let's try some queries and see cache hits!

In [ ]:
# Test:
print("="*60)
print("="*60)

query1 = "Can you check my order status?"
# query1 = "Where is my order?" 100% match
# query1 = "What is python?" CACHE MISS
result1 = search_cache(query1, threshold=0.2)

print(f"\n🔍 Query: {query1}")
if result1['hit']:
    print(f"✅ CACHE HIT!")
    print(f"   Matched: {result1['matched_question']}")
    print(f"   Distance: {result1['distance']:.4f}")
    print(f"   Similarity: {result1['similarity']:.2%}")
    print(f"\n💬 Answer: {result1['answer']}")
else:
    print(f"❌ CACHE MISS")


🔍 Query: Can you check my order status?
❌ CACHE MISS


## Experiment with Different Thresholds

See how threshold affects matching!

In [ ]:
print("="*60)
print("THRESHOLD EXPERIMENTS")
print("="*60)

test_query = "Can you check my order status?"
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

print(f"\n🔍 Query: {test_query}\n")

for threshold in thresholds:
    result = search_cache(test_query, threshold=threshold)
    status = "✅ HIT" if result['hit'] else "❌ MISS"
    print(f"Threshold {threshold:.1f}: {status} (distance: {result['distance']:.4f})")

THRESHOLD EXPERIMENTS

🔍 Query: Can you check my order status?

Threshold 0.1: ❌ MISS (distance: 0.2866)
Threshold 0.2: ❌ MISS (distance: 0.2865)
Threshold 0.3: ✅ HIT (distance: 0.2865)
Threshold 0.4: ✅ HIT (distance: 0.2865)
Threshold 0.5: ✅ HIT (distance: 0.2865)
Threshold 0.6: ✅ HIT (distance: 0.2865)


## Load the Necessary Libraries

In [1]:
from sentence_transformers import SentenceTransformer
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import List, Tuple, Optional, Dict, Callable
from dataclasses import dataclass, field
import time
import warnings
from pyprojroot import here
warnings.filterwarnings('ignore')

d:\work\new project\personal\help-parcel\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Implement the Data Loader**

In [7]:
print("\n📂 Setting up data infrastructure...\n")


@dataclass
class CacheResult:
    """Standardized result for cache lookups."""
    prompt: str
    response: str
    vector_distance: float
    cosine_similarity: float

    # Optional metadata for rerankers
    reranker_type: Optional[str] = None
    reranker_score: Optional[float] = None
    reranker_reason: Optional[str] = None


@dataclass
class CacheResults:
    """Container for query results with multiple potential matches."""
    query: str
    matches: List[CacheResult] = field(default_factory=list)

    def __repr__(self):
        return f"CacheResults(query='{self.query}', matches={len(self.matches)})"

    @property
    def hit(self) -> bool:
        return len(self.matches) > 0

    @property
    def best_match(self) -> Optional[CacheResult]:
        return self.matches[0] if self.matches else None


class FAQDataContainer:
    """
    Data container that manages FAQ data and test queries.

    This pattern keeps data loading and labeling logic in one place,
    making it easy to swap datasets or add new test cases.
    """

    def __init__(self, faq_path: str = here("data/faq_seed.csv"),
                 test_path: str = here("data/test_dataset.csv")):
        # Try with Python engine and proper escaping for CSV parsing
        try:
            self.faq_df = pd.read_csv(faq_path, engine='python')
            self.test_df = pd.read_csv(test_path, engine='python')
        except Exception as e:
            print(f"Error loading with python engine: {e}")
            # Fallback to reading with different parameters
            try:
                self.faq_df = pd.read_csv(faq_path, quotechar='"', escapechar='\\')
                self.test_df = pd.read_csv(test_path, quotechar='"', escapechar='\\')
            except Exception as e2:
                print(f"Error with escapechar: {e2}")
                # Final fallback - try reading with error handling
                self.faq_df = pd.read_csv(faq_path, on_bad_lines='skip', engine='python')
                self.test_df = pd.read_csv(test_path, on_bad_lines='skip', engine='python')

        print(f"✅ Loaded {len(self.faq_df)} FAQ entries")
        print(f"✅ Loaded {len(self.test_df)} test queries")
        print(f"   - Should hit cache: {self.test_df['cache_hit'].sum()}")
        print(f"   - Should miss cache: {(~self.test_df['cache_hit']).sum()}")

    def get_qa_pairs(self) -> List[Tuple[str, str]]:
        """Get all Q&A pairs for cache hydration."""
        return list(zip(self.faq_df["question"], self.faq_df["answer"]))

    def get_test_queries(self) -> List[str]:
        """Get all test queries."""
        return self.test_df["question"].tolist()

    def _resolve_question(self, q: str) -> Optional[str]:
        """Resolve a test query to its expected FAQ match."""
        matches = self.test_df[self.test_df["question"] == q]
        if len(matches) != 1:
            return None

        src_question_id = matches["src_question_id"].iloc[0]
        should_hit = matches["cache_hit"].iloc[0]

        if not should_hit:
            return None

        return self.faq_df.iloc[src_question_id]["question"]

    def label_cache_hits(self, cache_results: List) -> np.ndarray:
        """
        Label cache results as correct (True) or incorrect (False).

        A result is correct if:
        - Query should miss AND cache missed, OR
        - Query should hit AND cache hit the RIGHT question
        """
        results = []
        test_qs = set(self.test_df["question"].tolist())

        for res in cache_results:
            expected_hit = self._resolve_question(res.query)
            actual_hit = res.matches[0].prompt if res.matches else None

            # If the actual hit is also a test query, resolve it
            if actual_hit is not None and actual_hit in test_qs:
                actual_hit = self._resolve_question(actual_hit)

            results.append(expected_hit == actual_hit)

        return np.array(results)


📂 Setting up data infrastructure...



### **Load the data and Check Some Entries**

In [11]:
data = FAQDataContainer()

Error loading with python engine: Expected 4 fields in line 23, saw 5
Error with escapechar: Error tokenizing data. C error: Expected 4 fields in line 23, saw 5

✅ Loaded 10 FAQ entries
✅ Loaded 39 test queries
   - Should hit cache: 19
   - Should miss cache: 20


In [9]:
print("Number of FAQ entries:", data.faq_df.shape)
print("Column names in FAQ data:", data.faq_df.columns.tolist())
print("FAQ column types:", data.faq_df.dtypes)
data.faq_df.style

Number of FAQ entries: (10, 3)
Column names in FAQ data: ['id', 'question', 'answer']
FAQ column types: id          int64
question      str
answer        str
dtype: object


,id,question,answer
0,0,Where is my order?,You can track your order by using the tracking link sent to your email after shipment. Tracking updates may take 24–48 hours to appear.
1,1,How long does delivery take?,Standard delivery takes 3–7 business days depending on your location. Express shipping takes 1–3 business days.
2,2,Can I cancel my order?,"You can cancel your order only before it has been shipped. Once shipped, cancellation is no longer possible, but you may request a return."
3,3,How do I request a refund?,"To request a refund, go to your order details page and click 'Request Refund'. Refunds are processed within 5–10 business days after approval."
4,4,My item arrived damaged. What should I do?,Please report the issue within 48 hours of delivery and upload photos of the damaged item. We will process a replacement or refund after verification.
5,5,Can I change my shipping address?,You can change your shipping address only before the order is shipped by going to your order details and selecting 'Edit Address'.
6,6,Why is my tracking not updating?,"Tracking may take 24–48 hours to update after shipment. If there is no update after 3–5 days, please contact support."
7,7,Do you offer replacements?,"Yes, we offer replacements for damaged, defective, or incorrect items reported within 7 days of delivery."
8,8,Where can I find my order ID?,Your order ID can be found in your confirmation email and under the 'My Orders' section in your account dashboard.
9,9,What courier do you use?,We work with multiple shipping partners depending on your location. The courier details will appear once your order has been shipped.


In [12]:
print("Number of test queries along with the cache hit labels:", data.test_df.shape)
print("Column names in test data:", data.test_df.columns.tolist())
print("FAQ column types:", data.test_df.dtypes)
data.test_df.head(20).style

Number of test queries along with the cache hit labels: (39, 4)
Column names in test data: ['question', 'answer', 'src_question_id', 'cache_hit']
FAQ column types: question             str
answer               str
src_question_id    int64
cache_hit           bool
dtype: object


,question,answer,src_question_id,cache_hit
0,What's the process for getting my money back?,"To request a refund, visit your orders page and select **Request Refund**. Refunds are processed within 5–10 business days.",3,True
1,How can I request a refund for my purchase?,"To request a refund, go to your order details page and click 'Request Refund'. Refunds are processed within 5–10 business days.",3,True
2,What steps do I follow to return an item for money back?,"To request a refund, visit your orders page and select **Request Refund**. Refunds are processed within 5–10 business days.",3,True
3,Do I get money back if I cancel after shipping?,Refunds are only available for orders that have not yet been shipped or for approved returns after delivery.,3,True
4,Is refund available for change of mind?,Refunds are not guaranteed for change of mind unless the item is unused and within the return window.,3,True
5,What's your refund policy for digital products?,Digital products are non-refundable except in cases of technical defects or billing errors.,3,False
6,How much does it cost to process a refund?,"Refund processing is free for most payment methods, though bank transfers may incur a $5 processing fee.",3,False
7,Can I get a refund instantly?,Refunds are typically processed within 5–10 business days depending on your payment provider.,3,False
8,What's the weather like in Tokyo today?,I don't have access to real-time weather data. Please check a weather service for current forecasts.,3,False
9,How do I bake a chocolate cake?,"Mix flour, cocoa powder, sugar, eggs, and butter. Bake at 180°C for 30–35 minutes.",3,False


### **Load an Embedding Model**

In [13]:
print("🔄 Loading embedding model (this may take a moment)...")
# Using a well-known model - you can swap this for any embedding model
# encoder = SentenceTransformer("all-MiniLM-L6-v2")  # Dimension: 384,Fast & good quality
encoder = SentenceTransformer("all-mpnet-base-v2")  # dimension: 768, better quality but slower
print("✅ Model loaded!\n")

🔄 Loading embedding model (this may take a moment)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5635.94it/s]


✅ Model loaded!



In [14]:
# Demonstrate embeddings
demo_texts = [
    "I want my money back",
    "How do I get a refund?",
    "What's the weather today?"
]

print("📊 Embedding demonstration:")
print("-" * 60)
demo_embeddings = encoder.encode(demo_texts)
for text, emb in zip(demo_texts, demo_embeddings):
    print(f"'{text}'")
    print(f"   → Vector shape: {emb.shape}")
    print(f"   → First 5 dims: [{', '.join(f'{x:.3f}' for x in emb[:5])}...]")
    print()

📊 Embedding demonstration:
------------------------------------------------------------
'I want my money back'
   → Vector shape: (768,)
   → First 5 dims: [0.003, 0.118, -0.029, 0.005, -0.026...]

'How do I get a refund?'
   → Vector shape: (768,)
   → First 5 dims: [0.027, 0.042, -0.027, 0.038, -0.039...]

'What's the weather today?'
   → Vector shape: (768,)
   → First 5 dims: [-0.035, -0.037, -0.018, -0.033, 0.007...]



### **Implement the Functions For Vector Search**

In [15]:
def cosine_distance(a: np.ndarray, b: np.ndarray) -> float:
    """
    Calculate cosine distance between two vectors.

    Cosine Distance = 1 - Cosine Similarity

    Think of it like this:
    - Two arrows pointing the same direction = distance 0
    - Two arrows perpendicular = distance 1
    - Two arrows pointing opposite = distance 2
    """
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    cosine_similarity = dot_product / (norm_a * norm_b)
    return 1 - cosine_similarity


def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Calculate straight-line distance between two vectors."""
    return np.linalg.norm(a - b)


def cosine_distance_batch(embeddings: np.ndarray, query_emb: np.ndarray) -> np.ndarray:
    """Vectorized cosine distance for efficiency."""
    dot_products = np.dot(embeddings, query_emb)
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_emb)
    return 1 - (dot_products / norms)

In [16]:
# Demonstrate distance calculations
print("\n📊 Distance Comparison:\n")
print("-" * 70)

emb1, emb2, emb3 = demo_embeddings

print(f"Comparing: '{demo_texts[0]}' vs '{demo_texts[1]}'")
print(
    f"   Cosine Distance:    {cosine_distance(emb1, emb2):.4f}  (lower = more similar)")
print(f"   Euclidean Distance: {euclidean_distance(emb1, emb2):.4f}")
print()

print(f"Comparing: '{demo_texts[0]}' vs '{demo_texts[2]}'")
print(
    f"   Cosine Distance:    {cosine_distance(emb1, emb3):.4f}  (higher = less similar)")
print(f"   Euclidean Distance: {euclidean_distance(emb1, emb3):.4f}")
print()

print("💡 Notice: Refund questions are MUCH closer than weather question!")


📊 Distance Comparison:

----------------------------------------------------------------------
Comparing: 'I want my money back' vs 'How do I get a refund?'
   Cosine Distance:    0.3301  (lower = more similar)
   Euclidean Distance: 0.8126

Comparing: 'I want my money back' vs 'What's the weather today?'
   Cosine Distance:    0.9478  (higher = less similar)
   Euclidean Distance: 1.3768

💡 Notice: Refund questions are MUCH closer than weather question!


## **Comprehensive Test: Test Dataset vs FAQ Seed Lookup**

This test will evaluate how well our semantic search performs by:
1. Using queries from `test_dataset.csv` 
2. Looking up similar entries in `faq_seed.csv`
3. Comparing expected vs actual matches
4. Generating a detailed results table

In [56]:
def create_faq_cache_with_embeddings(data_container: FAQDataContainer, encoder: SentenceTransformer):
    """
    Create a FAQ cache with pre-computed embeddings for fast lookup.
    
    Returns:
        List of dicts with question, answer, id, and embedding
    """
    print("🔄 Creating FAQ cache with embeddings...")
    
    faq_cache = []
    qa_pairs = data_container.get_qa_pairs()
    
    # Get embeddings for all FAQ questions
    faq_questions = [q for q, a in qa_pairs]
    faq_embeddings = encoder.encode(faq_questions, show_progress_bar=True)
    
    # Build cache with embeddings
    for i, ((question, answer), embedding) in enumerate(zip(qa_pairs, faq_embeddings)):
        faq_cache.append({
            'id': i,
            'question': question,
            'answer': answer,
            'embedding': embedding
        })
    
    print(f"✅ FAQ cache created with {len(faq_cache)} entries")
    return faq_cache


def search_faq_cache(query: str, faq_cache: list, encoder: SentenceTransformer, top_k: int = 1):
    """
    Search FAQ cache for most similar questions using cosine distance.
    
    Returns:
        List of matches with similarity scores
    """
    # Get embedding for query
    query_embedding = encoder.encode([query])[0]
    
    # Calculate distances to all FAQ entries
    results = []
    for faq_entry in faq_cache:
        distance = cosine_distance(query_embedding, faq_entry['embedding'])
        similarity = 1 - distance  # Convert distance to similarity
        
        results.append({
            'faq_id': faq_entry['id'],
            'question': faq_entry['question'],
            'answer': faq_entry['answer'],
            'distance': distance,
            'similarity': similarity
        })
    
    # Sort by distance (closest first) and return top_k
    results.sort(key=lambda x: x['distance'])
    return results[:top_k]


# Create the FAQ cache
faq_cache = create_faq_cache_with_embeddings(data, encoder)

🔄 Creating FAQ cache with embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.33it/s]

✅ FAQ cache created with 10 entries


In [57]:
def run_comprehensive_test(data_container: FAQDataContainer, faq_cache: list, encoder: SentenceTransformer):
    """
    Run comprehensive test comparing test queries against FAQ cache.
    
    Returns:
        DataFrame with detailed results
    """
    print("🧪 Running comprehensive test...")
    print("-" * 60)
    
    results = []
    test_queries = data_container.get_test_queries()
    
    for i, test_query in enumerate(tqdm(test_queries, desc="Processing test queries")):
        # Get expected result from test data
        test_row = data_container.test_df[data_container.test_df['question'] == test_query].iloc[0]
        expected_faq_id = test_row['src_question_id']
        should_hit_cache = test_row['cache_hit']
        expected_answer = test_row['answer']
        
        # Get expected FAQ entry
        if should_hit_cache:
            expected_faq_entry = data_container.faq_df.iloc[expected_faq_id]
            expected_faq_question = expected_faq_entry['question']
            expected_faq_answer = expected_faq_entry['answer']
        else:
            expected_faq_question = "N/A (Should Miss)"
            expected_faq_answer = "N/A (Should Miss)"
        
        # Search FAQ cache
        matches = search_faq_cache(test_query, faq_cache, encoder, top_k=1)
        best_match = matches[0] if matches else None
        
        if best_match:
            matched_faq_id = best_match['faq_id']
            matched_question = best_match['question']
            matched_answer = best_match['answer']
            similarity_score = best_match['similarity']
        else:
            matched_faq_id = None
            matched_question = "No match found"
            matched_answer = "No match found"
            similarity_score = 0.0
        
        # Determine cache hit based on similarity threshold (0.3)
        cache_hit = similarity_score >= 0.3
        
        # Store results
        results.append({
            'Test_ID': i,
            'Question': test_query,
            'Should_Hit_Cache': should_hit_cache,
            'Cache_Hit': cache_hit,
            'Expected_FAQ_ID': expected_faq_id if should_hit_cache else 'N/A',
            'Matched_FAQ_ID': matched_faq_id,
            'Expected_FAQ_Question': expected_faq_question,
            'Matched_FAQ_Question': matched_question,
            'Expected_Answer': expected_faq_answer,
            'Matched_Answer': matched_answer,
            'Similarity_Score': similarity_score
        })
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Calculate cache alignment statistics
    total_tests = len(results_df)
    cache_alignment_correct = len(results_df[results_df['Should_Hit_Cache'] == results_df['Cache_Hit']])
    cache_alignment_rate = cache_alignment_correct / total_tests * 100
    
    cache_hits = len(results_df[results_df['Cache_Hit'] == True])
    cache_misses = len(results_df[results_df['Cache_Hit'] == False])
    
    print(f"\n📊 Test Results Summary:")
    print(f"   Total Tests: {total_tests}")
    print(f"   ✅ Cache Behavior Correct: {cache_alignment_correct} ({cache_alignment_rate:.1f}%)")
    print(f"   ❌ Cache Behavior Incorrect: {total_tests - cache_alignment_correct} ({100 - cache_alignment_rate:.1f}%)")
    print(f"\n📊 Cache Performance:")
    print(f"   Cache Hits (≥0.3 similarity): {cache_hits}/{total_tests} ({cache_hits/total_tests*100:.1f}%)")
    print(f"   Cache Misses (<0.3 similarity): {cache_misses}/{total_tests} ({cache_misses/total_tests*100:.1f}%)")
    
    return results_df

# Run the comprehensive test
results_df = run_comprehensive_test(data, faq_cache, encoder)

🧪 Running comprehensive test...
------------------------------------------------------------


Processing test queries: 100%|██████████| 39/39 [00:01<00:00, 37.49it/s]


📊 Test Results Summary:
   Total Tests: 39
   ✅ Cache Behavior Correct: 35 (89.7%)
   ❌ Cache Behavior Incorrect: 4 (10.3%)

📊 Cache Performance:
   Cache Hits (≥0.3 similarity): 21/39 (53.8%)
   Cache Misses (<0.3 similarity): 18/39 (46.2%)


### **Display Detailed Results Table**

In [58]:
# Display the full results table
print("📋 Detailed Test Results:")
print("=" * 100)

# Create a more readable version for display
display_df = results_df.copy()

# Truncate long text for better display
def truncate_text(text, max_length=50):
    return text if len(str(text)) <= max_length else str(text)[:max_length] + "..."

display_df['Question'] = display_df['Question'].apply(lambda x: truncate_text(x, 60))
display_df['Expected_Answer'] = display_df['Expected_Answer'].apply(lambda x: truncate_text(x, 50))
display_df['Matched_Answer'] = display_df['Matched_Answer'].apply(lambda x: truncate_text(x, 50))

# Round similarity scores
display_df['Similarity_Score'] = display_df['Similarity_Score'].round(3)

# Select key columns for display
columns_to_show = [
    'Test_ID', 'Question', 'Expected_FAQ_ID', 'Matched_FAQ_ID', 
    'Cache_Hit', 'Similarity_Score'
]

print(f"Showing {len(display_df)} test results:")
display_df[columns_to_show].style

📋 Detailed Test Results:
Showing 39 test results:


,Test_ID,Question,Expected_FAQ_ID,Matched_FAQ_ID,Cache_Hit,Similarity_Score
0,0,What's the process for getting my money back?,3,3,True,0.819000
1,1,How can I request a refund for my purchase?,3,3,True,0.913000
2,2,What steps do I follow to return an item for money back?,3,3,True,0.714000
3,3,Do I get money back if I cancel after shipping?,3,2,True,0.664000
4,4,Is refund available for change of mind?,3,3,True,0.540000
5,5,What's your refund policy for digital products?,N/A,3,True,0.424000
6,6,How much does it cost to process a refund?,N/A,3,True,0.437000
7,7,Can I get a refund instantly?,N/A,3,True,0.661000
8,8,What's the weather like in Tokyo today?,N/A,1,False,0.128000
9,9,How do I bake a chocolate cake?,N/A,1,False,0.165000


### **Analyze Failed Cases**

In [54]:
# Analyze cache behavior correctness
# Check for cases where cache behavior doesn't align with expectations
misaligned_cases = results_df[
    (results_df['Should_Hit_Cache'] != results_df['Cache_Hit'])
]

if len(misaligned_cases) > 0:
    print(f"🔍 Analyzing {len(misaligned_cases)} cache behavior misalignments:")
    print("=" * 80)
    
    # Cases that should hit cache but didn't
    should_hit_but_missed = misaligned_cases[
        (misaligned_cases['Should_Hit_Cache'] == True) & 
        (misaligned_cases['Cache_Hit'] == False)
    ]
    
    if len(should_hit_but_missed) > 0:
        print(f"\n❌ Should Hit Cache But Missed ({len(should_hit_but_missed)} cases):")
        print("These queries should have found similar FAQs but didn't reach the 0.3 threshold")
        print("-" * 60)
        for idx, row in should_hit_but_missed.iterrows():
            print(f"\n🎯 Test #{row['Test_ID']}:")
            print(f"   Query: '{row['Question']}'")
            print(f"   Expected FAQ ID: {row['Expected_FAQ_ID']}")
            print(f"   Matched FAQ ID: {row['Matched_FAQ_ID']}")
            print(f"   Similarity Score: {row['Similarity_Score']:.3f} (< 0.3 threshold)")
            print(f"   Expected: '{row['Expected_FAQ_Question'][:60]}...'")
            print(f"   Matched:  '{row['Matched_FAQ_Question'][:60]}...'")
    
    # Cases that shouldn't hit cache but did
    should_miss_but_hit = misaligned_cases[
        (misaligned_cases['Should_Hit_Cache'] == False) & 
        (misaligned_cases['Cache_Hit'] == True)
    ]
    
    if len(should_miss_but_hit) > 0:
        print(f"\n⚠️ Should Miss Cache But Hit ({len(should_miss_but_hit)} cases):")
        print("These queries should not match FAQs but exceeded the 0.3 threshold")
        print("-" * 60)
        for idx, row in should_miss_but_hit.iterrows():
            print(f"\n🚫 Test #{row['Test_ID']}:")
            print(f"   Query: '{row['Question']}'")
            print(f"   Matched FAQ ID: {row['Matched_FAQ_ID']}")
            print(f"   Similarity Score: {row['Similarity_Score']:.3f} (≥ 0.3 threshold)")
            print(f"   Matched:  '{row['Matched_FAQ_Question'][:60]}...'")

else:
    print("🎉 Perfect cache alignment! All cache hits/misses match expectations.")

🔍 Analyzing 4 cache behavior misalignments:

❌ Should Hit Cache But Missed (1 cases):
These queries should have found similar FAQs but didn't reach the 0.3 threshold
------------------------------------------------------------

🎯 Test #22:
   Query: 'What should I do if I can't log in?'
   Expected FAQ ID: 1
   Matched FAQ ID: 4
   Similarity Score: 0.282 (< 0.3 threshold)
   Expected: 'How long does delivery take?...'
   Matched:  'My item arrived damaged. What should I do?...'

⚠️ Should Miss Cache But Hit (3 cases):
These queries should not match FAQs but exceeded the 0.3 threshold
------------------------------------------------------------

🚫 Test #5:
   Query: 'What's your refund policy for digital products?'
   Matched FAQ ID: 3
   Similarity Score: 0.424 (≥ 0.3 threshold)
   Matched:  'How do I request a refund?...'

🚫 Test #6:
   Query: 'How much does it cost to process a refund?'
   Matched FAQ ID: 3
   Similarity Score: 0.437 (≥ 0.3 threshold)
   Matched:  'How do I request 

### **Export Full Results**

Save the complete results to a CSV file for further analysis.

In [ ]:
# Save full results to CSV
output_file = "semantic_search_test_results.csv"
results_df.to_csv(output_file, index=False)

print(f"💾 Full results saved to: {output_file}")
print(f"📊 File contains {len(results_df)} test results with all details")

# Display final summary
print(f"\n🏆 Final Test Summary:")
print("=" * 50)
total = len(results_df)
cache_alignment_rate = len(results_df[results_df['Should_Hit_Cache'] == results_df['Cache_Hit']]) / total * 100
print(f"Cache Alignment Rate: {cache_alignment_rate:.1f}% ({len(results_df[results_df['Should_Hit_Cache'] == results_df['Cache_Hit']])}/{total})")

# Show average similarity scores for different categories
hit_cache_results = results_df[results_df['Should_Hit_Cache'] == True]
miss_cache_results = results_df[results_df['Should_Hit_Cache'] == False]

if len(hit_cache_results) > 0:
    avg_sim_hit = hit_cache_results['Similarity_Score'].mean()
    print(f"Average similarity for cache hits: {avg_sim_hit:.3f}")

if len(miss_cache_results) > 0:
    avg_sim_miss = miss_cache_results['Similarity_Score'].mean()
    print(f"Average similarity for cache misses: {avg_sim_miss:.3f}")

# Show cache hit statistics
cache_hits = len(results_df[results_df['Cache_Hit'] == True])
cache_misses = len(results_df[results_df['Cache_Hit'] == False])
print(f"\n📊 Cache Performance:")
print(f"Cache Hits (≥0.3 similarity): {cache_hits}/{total} ({cache_hits/total*100:.1f}%)")
print(f"Cache Misses (<0.3 similarity): {cache_misses}/{total} ({cache_misses/total*100:.1f}%)")

💾 Full results saved to: semantic_search_test_results.csv
📊 File contains 39 test results with all details

🏆 Final Test Summary:
Success Rate: 64.1% (25/39)
Average similarity for cache hits: 0.593
Average similarity for cache misses: 0.171

📊 Cache Performance:
Cache Hits (≥0.3 similarity): 21/39 (53.8%)
Cache Misses (<0.3 similarity): 18/39 (46.2%)
